# 5. Evaluate Single Harnesses Against a MemoRizz Panel

This notebook explains and reproduces the corrected `structured_adaptive_v1` evaluation. It compares direct Codex, direct Claude Code, and MemoRizz panels backed by Filesystem and Oracle AI Database. The panel is a **coordinated topology**: a MemAgent owns memory and policy while Codex is the primary worker and Claude Code is invoked only when host-side coverage validation finds a gap.

The evaluator implementation lives in one maintained Python runner rather than being copied into notebook cells. The safe path below reads the sanitized two-repeat artifact and makes no external calls.

## Experimental topology

```mermaid
flowchart LR
    F[Identical read-only fixture] --> C[Codex only]
    F --> A[Claude Code only]
    F --> MF[MemoRizz panel - Filesystem]
    F --> MO[MemoRizz panel - Oracle]
    MF --> P1[Codex primary]
    MO --> P2[Codex primary]
    P1 & P2 --> G{Host coverage gate}
    G -- missing criteria --> CL[Claude fallback]
    G -- complete --> U[Deterministic union]
    CL --> U
    C & A & U --> V[Host verification]
    V --> J[Identity-blinded judge]
```

Every executed Codex or Claude process receives the same per-harness budget. Adaptive routing is the treatment: a panel may execute fewer workers when its typed primary result covers every required criterion.

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

from IPython.display import Markdown, display

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'memorizz').exists():
            return candidate
    raise RuntimeError('Open this notebook from the MemoRizz checkout or a child directory.')

REPO_ROOT = find_repo_root()
ARTIFACT_PATH = REPO_ROOT / 'eval' / 'results' / '2026-08-22-metaharness-provider-comparison-optimized.json'
RUNNER_PATH = REPO_ROOT / 'eval' / 'metaharness' / 'provider_comparison.py'
artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
assert artifact['schema_version'] == 'memorizz.metaharness.provider-comparison.v2'
assert artifact['status'] == 'valid'
assert artifact['paper_comparable'] is False
print({'artifact': str(ARTIFACT_PATH.relative_to(REPO_ROOT)), 'protocol': artifact['scope']['protocol'], 'repeats': artifact['scope']['repeats'], 'paper_comparable': artifact['paper_comparable']})

## Fairness and validity controls

- identical fixture, requirements, host verification, and read-only/no-network policy;
- cold, isolated memory and thread scopes for every arm and repeat;
- counterbalanced execution order;
- the same structured finding contract and per-harness resource envelope;
- provider-neutral evidence-content fingerprints across Filesystem and Oracle;
- identity-blinded judge inputs; and
- fail-closed rejection of failed, unverified, ungrounded, or schema-invalid results.

This is an engineering evaluation, not a leaderboard run. Two repeats on one synthetic fixture estimate integration behavior; they do not establish population-level model quality.

In [ ]:
labels = {
    'codex_only': 'Codex only',
    'claude_only': 'Claude Code only',
    'memorizz_panel_filesystem': 'MemoRizz panel (Filesystem)',
    'memorizz_panel_oracle': 'MemoRizz panel (Oracle AI Database)',
}
header = '| System | Judge | Gold | Unsupported | Latency | Cost/task | Grounded |\n|---|---:|---:|---:|---:|---:|---:|'
rows = []
for result in artifact['results']:
    rows.append(
        f"| {labels[result['arm']]} | {result['judge_score_mean']:.1f} ± {result['judge_score_sample_stdev']:.1f} | "
        f"{result['gold_findings_mean']:.0f}/6 | {result['unsupported_claims_mean']:.1f} | "
        f"{result['wall_latency_ms_mean']/1000:.1f}s | ${result['cost_usd_mean']:.6f} | "
        f"{result['grounded_rate']*100:.0f}% |"
    )
display(Markdown(header + '\n' + '\n'.join(rows)))

## Reproduce through the maintained runner

The runner creates both memory providers, performs Oracle and harness preflight before spend, writes the fixture, seeds equivalent scoped evidence, initializes every MemAgent and external harness, executes counterbalanced arms, validates every result, invokes the blinded judge, writes one artifact, and transactionally cleans synthetic Oracle scopes.

Keeping that lifecycle in `eval/metaharness/provider_comparison.py` prevents notebook code and release code from drifting. Notebook 06 deliberately takes the opposite educational approach: it constructs three independent adapters in cells so you can see exactly how a native MemAgent becomes a harness.

In [ ]:
RUN_LIVE = os.getenv('MEMORIZZ_RUN_HARNESS_EVALUATION') == '1'
output = Path(tempfile.gettempdir()) / 'memorizz-panel-comparison-notebook.json'
command = [sys.executable, str(RUNNER_PATH), '--repeats', '2', '--seed', '20260822', '--output', str(output)]
if RUN_LIVE:
    command.append('--execute')
    subprocess.run(command, cwd=REPO_ROOT, check=True)
    print('New raw artifact:', output)
else:
    print('Paid execution skipped. Set MEMORIZZ_RUN_HARNESS_EVALUATION=1 before starting Jupyter to opt in.')
    print('Command that would run:', ' '.join(command + ['--execute']))

## Interpretation boundary

The optimized panel avoided both Claude fallback and model synthesis in these repeats, which explains its low observed cost. That is not a promise that fallback remains zero on broader work. Report quality, cost, latency, grounding, verification, and fallback rate separately. For an independent native-MemAgent arm—where MemoRizz supplies the reasoning loop rather than coordinating Codex and Claude—continue to notebook 06.